# Hi-C data analysis workshop (part 1: hicstuff pipeline)

Aurèle Piazza, Romain Bulteau

25/03/2026

---

## Overview
In this section, we will generate a Hi-C contact matrix from raw reads, and go over the important steps of the pipeline.

More specifically:
  - Run a pipeline from raw data to analysis-ready contact matrices  
  - Understand important steps
    - read digestion
    - filtering steps 
    - binning
    - normalization (ICED balancing)
  - Understand output formats and plots
    - graal matrix
    - cool/mcool formats
    - event distributions
    - genome fragmentation

## Setup

### Working directory
Create a `HiC_analysis` folder in your home. This will house your scripts and notebooks for the workshop.

In a terminal:
```sh
cd ~/
mkdir -p HiC_analysis/
```

Copy the notebooks and scripts from `workshop_notebooks/` to your folder.

```sh
cp /PATH_TO_NOTEBOOKS/workshop_notebooks/* ~/HiC_analysis/
```


Check you have all the files, as listed below:

```sh
ls -1 ~/HiC_analysis
# HiC_workshop_part1.ipynb
# HiC_workshop_part2.ipynb
# HiC_workshop_part3.ipynb
# reads_to_hic.sh
```


### Conda
Activate the conda environment containing the tools used in the workshop, and check you can run the `hicstuff` command: 

```sh
conda activate hic
hicstuff --version
# 3.2.4
```

## Launching the pipeline

Open the `reads_to_hic.sh` bash script in a text editor.

Check the 'Input parameters' section at the start of the script (shown below) and fill out the paths to sample and reference data as instructed.

```sh
## Input parameters ------------------------------------------------------------

thread=4              # number of threads
mode="normal"         # 'parasplit' or 'cutsite' or 'normal' (for no digestion)
enzymes="DpnII,HinfI" # enzymes to digest the reads
quality=20            # alignment quality

fastqdir='/path/to/data/dir'    # path to fastq file directory
genomedir='/path/to/genome/dir' # path to bowtie genome index directory
outputdir='/localtmp/'          # path to pipeline output directory
outdir_digest="${fastqdir}"     # path to store digested reads

SAMP=("AD281_0.02_subsamp_parasplit") #list of sample to run, 
```

Launch the script from a terminal where the conda environment is active:

```sh
bash reads_to_hic.sh
```


## Pipeline overview

1. Digest reads (parasplit / cutsite)
2. Align fragments to reference genome (bowtie2, by hicstuff)
3. Pair uniquely-mapped fragments (hicstuff)
4. Filter uninformative reads
    - PCR duplicates (identical read mapping positions)
    - Loops (fragments religated to themselves)
    - "Uncut" (and religated) fragments (consecutive restriction fragment contacts)
5. Binning (cooler)
6. Normalization / balancing (cooler / hicstuff / ...)


## Pipeline outputs

Once the pipeline has finished running, go to your output folder and check its contents. You should see the following:

```
output_folder/
└── AD281_0.02_subsamp/
    ├── abs_fragments_contacts_weighted.txt
    ├── AD281_0.02_subsamp_S288c_DSB_chr3_rDNA_parasplit_q20_v324_1kb.chr.tsv
    ├── AD281_0.02_subsamp_S288c_DSB_chr3_rDNA_parasplit_q20_v324_1kb.frags.tsv
    ├── AD281_0.02_subsamp_S288c_DSB_chr3_rDNA_parasplit_q20_v324_1kb.mat.tsv
    ├── Cool
    │   ├── AD281_0.02_subsamp_S288c_DSB_chr3_rDNA_parasplit_q20_v324_1kb.cool
    │   └── AD281_0.02_subsamp_S288c_DSB_chr3_rDNA_parasplit_q20_v324.mcool
    ├── distance_law.txt
    ├── fragments_list.txt
    ├── hicstuff_20260107152429.log
    ├── info_contigs.txt
    ├── plots
    │   ├── distance_law.pdf
    │   ├── event_distance.pdf
    │   ├── event_distribution.pdf
    │   └── frags_hist.pdf
    ├── tmp
    │   ├── for.bam
    │   ├── genome.fa.gz
    │   ├── rev.bam
    │   ├── valid_idx_filtered.pairs
    │   ├── valid_idx.pairs
    │   └── valid.pairs
    └── valid_idx_pcrfree.pairs.gz

```

Let's break down the contents.

### mat, frags, chr

We have 3 main output file types output by `hicstuff`: contact matrices, genome fragment coordinates, and chromosome size info.
`abs_fragments_contacts_weighted.txt`, `fragments_list.txt`, and `info_contigs.txt` respectively correspond to these outputs.


We will have a look inside each file with `head <file>`.

#### mat
```sh
head abs_fragments_contacts_weighted.txt
```
```
74860	74860	872872
0	1	1
0	3	1
0	16	1
0	22	2
0	27	1
0	28	1
```

This file is in GRAAL format, with the first line corresponding to the matrix dimensions and the sum of contacts, and following lines indicating bin coordinates and number of contacts.
Note that not all fragment combination are stored to save space.

#### frags
```sh
head fragments_list.txt
```
``` 
id	chrom	start_pos	end_pos	size	gc_content
1	chr1	0	336	336	0.0051190476190476186
2	chr1	336	476	140	0.004357142857142857
3	chr1	476	509	33	0.005151515151515152
4	chr1	509	1149	640	0.00359375
5	chr1	1149	1410	261	0.0036781609195402297
6	chr1	1410	1492	82	0.00475609756097561
```

Each fragment corresponds to a genome bin flanked by restriction sites for the enzymes used for digestion.
Size of the fragments varies with uneven positioning of the restriction sites.


#### chr
```sh
head info_contigs.txt
```
``` 
contig	length	n_frags	cumul_length
chr1	230218	1358	0
chr2	813184	4966	1358
chr3	316620	1933	6324
chr4	1531933	9698	8257
chr5	578613	3486	17955
chr6	270161	1734	21441
```

### plots

`hicstuff` outputs 3 diagnostics plots:

- `frag_hists.pdf` : distribution of genomic fragment sizes between restriction sites.
- `event_distribution.pdf` : distribution of restriction fragment pair configurations in reads (e.g. loops, uncut/undigested).
- `event_distance.pdf` : thresholds for removal of uninformative configurations.

### distance_law

The distance law, or distance-dependence relationship of contacts, or $P(s)$, illustrates the link between the distance separating two (intrachromosomal) points, and their contact frequency.
`hicstuff` stores the computed distance laws per chromosome in `distance_law.txt`, and its corresponding plot in `plots/distance_laws.pdf` 

### valid_idx_pcrfree.pairs.gz

This file contains the mapping information of each valid read pair. 
Most HiC analysis tools go through this format before producing more compact (but lossy) formats (such as GRAAL or Cool).

```sh
zcat valid_idx_pcrfree.pairs.gz | head -n10000 | tail  # head+tail to check mid-file
```
```
A00709:366:HKW2GDSX3:2:1253:22001:9596:01:01	chr1	88152	chr1	97913	+	-	501	558
A00709:366:HKW2GDSX3:2:1153:27615:10300:01:01	chr1	88196	chr1	94453	+	+	501	542
A00627:386:HKG5KDSX3:2:2475:19678:4413:01:01	chr1	88197	chr1	89798	-	-	501	513
A00709:366:HKW2GDSX3:2:2427:19822:35258	chr1	88202	chr1	101502	+	+	501	574
```

### Cool files

Cool is a binary sparse-matrix format used to store genomic contact data.
We generated a 1kb-resolution binned cool and a multi-resolution binned cool (`.mcool`) with our pipeline, which we'll use in the next part of this workshop. 

